# JSON Basics — 04: Nested JSON

Real-world JSON is almost always nested — objects within objects,
arrays of objects, mixed structures. This notebook covers all patterns.

Topics: struct access, explode arrays, flatten, JSON strings embedded in columns,
from_json/to_json, get_json_object for selective extraction.


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/json_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('json-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 20:48:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


## Step 1 — Create Nested JSON Dataset



In [2]:

import json as pyjson, random, datetime
random.seed(42)

events = []
for i in range(10_000):
    event = {
        "event_id": i+1,
        "user": {
            "id": random.randint(1,500),
            "name": f"User_{random.randint(1,500)}",
            "address": {
                "city": random.choice(["NYC","LA","Chicago","Houston","Phoenix"]),
                "country": random.choice(["US","UK","DE","FR"])
            }
        },
        "items": [
            {"sku": f"SKU{j}", "price": round(random.uniform(5,200),2), "qty": random.randint(1,5)}
            for j in range(random.randint(1,4))
        ],
        "totals": {
            "subtotal": round(random.uniform(10,800),2),
            "tax":      round(random.uniform(1,80),2),
            "shipping": round(random.uniform(0,25),2)
        },
        "tags": random.sample(["vip","mobile","returning","first_time","promo"], random.randint(0,3)),
        "metadata": pyjson.dumps({"session": f"sess_{i}", "ab_test": f"v{random.randint(1,3)}"}),
        "created_at": str(datetime.datetime(2024,random.randint(1,12),random.randint(1,28),
                                            random.randint(0,23),random.randint(0,59)))
    }
    events.append(event)

nested_path = f"{DATA_DIR}/events_nested.json"
with open(nested_path,"w") as f:
    for e in events: f.write(pyjson.dumps(e)+"\n")
print(f"Created: {len(events):,} nested events")
df = spark.read.json(nested_path)
df.printSchema()


Created: 10,000 nested events


[Stage 0:>                                                          (0 + 1) / 1]

root
 |-- created_at: string (nullable = true)
 |-- event_id: long (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- price: double (nullable = true)
 |    |    |-- qty: long (nullable = true)
 |    |    |-- sku: string (nullable = true)
 |-- metadata: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- totals: struct (nullable = true)
 |    |-- shipping: double (nullable = true)
 |    |-- subtotal: double (nullable = true)
 |    |-- tax: double (nullable = true)
 |-- user: struct (nullable = true)
 |    |-- address: struct (nullable = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)



## Step 2 — Accessing Nested Fields



In [3]:

df = spark.read.json(f"{DATA_DIR}/events_nested.json")

print("Dot notation for nested struct access:")
df.select(
    "event_id",
    "user.id",
    "user.name",
    "user.address.city",
    "user.address.country",
    "totals.subtotal",
    "totals.tax",
).show(5)

# Filter on nested field
print("Filter on nested field:")
df.filter(F.col("user.address.country") == "US") \
  .select("event_id","user.name","user.address.city","totals.subtotal") \
  .show(5)


Dot notation for nested struct access:
+--------+---+--------+-------+-------+--------+-----+
|event_id| id|    name|   city|country|subtotal|  tax|
+--------+---+--------+-------+-------+--------+-----+
|       1|328| User_58|    NYC|     DE|   78.68|34.33|
|       2|367|User_333|Phoenix|     FR|  609.46|13.61|
|       3|184|User_434|Chicago|     DE|  108.61|73.86|
|       4|339|User_117|Chicago|     US|   138.5|29.07|
|       5|126| User_84|Houston|     FR|  190.95| 3.54|
+--------+---+--------+-------+-------+--------+-----+
only showing top 5 rows
Filter on nested field:
+--------+--------+-------+--------+
|event_id|    name|   city|subtotal|
+--------+--------+-------+--------+
|       4|User_117|Chicago|   138.5|
|       8|User_257|     LA|  436.11|
|      11|User_113|    NYC|  689.11|
|      13|User_373|Chicago|  154.96|
|      14|User_429|    NYC|  722.93|
+--------+--------+-------+--------+
only showing top 5 rows


## Step 3 — Explode Arrays of Objects



In [4]:

df = spark.read.json(f"{DATA_DIR}/events_nested.json")

# Explode items array — one row per item
print("explode(items) — one row per cart item:")
df_items = df.select(
    "event_id",
    "user.id",
    F.explode("items").alias("item")
).select(
    "event_id","id",
    F.col("item.sku").alias("sku"),
    F.col("item.price").alias("price"),
    F.col("item.qty").alias("qty"),
    (F.col("item.price") * F.col("item.qty")).alias("line_total")
)
print(f"After explode: {df_items.count():,} rows (was {df.count():,})")
df_items.show(8)

# posexplode — includes array position index
print("\nposexplode(items) — with position:")
df.select("event_id", F.posexplode("items").alias("pos","item")) \
  .select("event_id","pos","item.sku","item.price").show(6)

# explode tags (simple string array)
print("\nexplode_outer(tags) — keeps empty arrays as null row:")
df.select("event_id", F.explode_outer("tags").alias("tag")).show(8)


explode(items) — one row per cart item:


After explode: 24,957 rows (was 10,000)
+--------+---+----+------+---+------------------+
|event_id| id| sku| price|qty|        line_total|
+--------+---+----+------+---+------------------+
|       1|328|SKU0| 48.53|  1|             48.53|
|       1|328|SKU1|136.96|  5| 684.8000000000001|
|       2|367|SKU0|  92.6|  3|277.79999999999995|
|       2|367|SKU1|162.84|  1|            162.84|
|       3|184|SKU0| 147.3|  5|             736.5|
|       4|339|SKU0|173.96|  4|            695.84|
|       4|339|SKU1|  59.2|  3|177.60000000000002|
|       5|126|SKU0|197.96|  5| 989.8000000000001|
+--------+---+----+------+---+------------------+
only showing top 8 rows

posexplode(items) — with position:
+--------+---+----+------+
|event_id|pos| sku| price|
+--------+---+----+------+
|       1|  0|SKU0| 48.53|
|       1|  1|SKU1|136.96|
|       2|  0|SKU0|  92.6|
|       2|  1|SKU1|162.84|
|       3|  0|SKU0| 147.3|
|       4|  0|SKU0|173.96|
+--------+---+----+------+
only showing top 6 rows

explo

## Step 4 — from_json / to_json: JSON Strings in Columns



In [5]:

df = spark.read.json(f"{DATA_DIR}/events_nested.json")

# metadata column is a JSON string inside the JSON — very common in Kafka
print("metadata column is a JSON string:")
df.select("event_id","metadata").show(3, truncate=False)

# Parse embedded JSON string with from_json
meta_schema = StructType([
    StructField("session",  StringType()),
    StructField("ab_test",  StringType()),
])
df_parsed = df.withColumn("meta_parsed", F.from_json(F.col("metadata"), meta_schema)) \
              .select("event_id",
                      F.col("meta_parsed.session").alias("session"),
                      F.col("meta_parsed.ab_test").alias("ab_test"))
print("\nAfter from_json:")
df_parsed.show(5)

# to_json: convert struct column back to JSON string
df_struct = df.select("event_id","user","totals")
df_json_str = df_struct.withColumn("user_json",   F.to_json(F.col("user"))) \
                       .withColumn("totals_json",  F.to_json(F.col("totals"))) \
                       .select("event_id","user_json","totals_json")
print("\nto_json — struct to JSON string:")
df_json_str.show(3, truncate=False)


metadata column is a JSON string:
+--------+--------------------------------------+
|event_id|metadata                              |
+--------+--------------------------------------+
|1       |{"session": "sess_0", "ab_test": "v3"}|
|2       |{"session": "sess_1", "ab_test": "v2"}|
|3       |{"session": "sess_2", "ab_test": "v3"}|
+--------+--------------------------------------+
only showing top 3 rows

After from_json:
+--------+-------+-------+
|event_id|session|ab_test|
+--------+-------+-------+
|       1| sess_0|     v3|
|       2| sess_1|     v2|
|       3| sess_2|     v3|
|       4| sess_3|     v3|
|       5| sess_4|     v3|
+--------+-------+-------+
only showing top 5 rows

to_json — struct to JSON string:
+--------+------------------------------------------------------------------------+------------------------------------------------+
|event_id|user_json                                                               |totals_json                                     |
+------

## Step 5 — get_json_object: Selective Extraction



In [6]:

df = spark.read.json(f"{DATA_DIR}/events_nested.json")

# get_json_object extracts a single value from JSON string using JSONPath
# Useful when you only need 1-2 fields from a large embedded JSON
print("get_json_object on metadata column:")
df.select(
    "event_id",
    F.get_json_object(F.col("metadata"), "$.session").alias("session"),
    F.get_json_object(F.col("metadata"), "$.ab_test").alias("ab_test"),
).show(5)

# json_tuple: extract multiple values at once (more efficient than multiple get_json_object)
print("\njson_tuple — multiple keys at once:")
df.select("event_id",
    F.json_tuple(F.col("metadata"), "session","ab_test").alias("session","ab_test")
).show(5)


get_json_object on metadata column:
+--------+-------+-------+
|event_id|session|ab_test|
+--------+-------+-------+
|       1| sess_0|     v3|
|       2| sess_1|     v2|
|       3| sess_2|     v3|
|       4| sess_3|     v3|
|       5| sess_4|     v3|
+--------+-------+-------+
only showing top 5 rows

json_tuple — multiple keys at once:
+--------+-------+-------+
|event_id|session|ab_test|
+--------+-------+-------+
|       1| sess_0|     v3|
|       2| sess_1|     v2|
|       3| sess_2|     v3|
|       4| sess_3|     v3|
|       5| sess_4|     v3|
+--------+-------+-------+
only showing top 5 rows


## Step 6 — Flatten for Analytics



In [7]:

df = spark.read.json(f"{DATA_DIR}/events_nested.json")

meta_schema = StructType([StructField("session",StringType()),StructField("ab_test",StringType())])

df_flat = df \
    .withColumn("meta", F.from_json("metadata", meta_schema)) \
    .select(
        "event_id",
        F.col("user.id").alias("user_id"),
        F.col("user.address.city").alias("city"),
        F.col("user.address.country").alias("country"),
        F.col("totals.subtotal").alias("subtotal"),
        F.col("totals.tax").alias("tax"),
        F.size("items").alias("item_count"),
        F.size("tags").alias("tag_count"),
        F.col("meta.ab_test").alias("ab_test"),
        F.to_timestamp(F.col("created_at")).alias("created_at"),
    )
print("Fully flattened for analytics:")
df_flat.printSchema()
df_flat.show(5)
df_flat.write.mode("overwrite").parquet(f"{DATA_DIR}/events_flat.parquet")
print(f"\nSaved as Parquet: {spark.read.parquet(f'{DATA_DIR}/events_flat.parquet').count():,} rows")


Fully flattened for analytics:
root
 |-- event_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- subtotal: double (nullable = true)
 |-- tax: double (nullable = true)
 |-- item_count: integer (nullable = false)
 |-- tag_count: integer (nullable = false)
 |-- ab_test: string (nullable = true)
 |-- created_at: timestamp (nullable = true)

+--------+-------+-------+-------+--------+-----+----------+---------+-------+-------------------+
|event_id|user_id|   city|country|subtotal|  tax|item_count|tag_count|ab_test|         created_at|
+--------+-------+-------+-------+--------+-----+----------+---------+-------+-------------------+
|       1|    328|    NYC|     DE|   78.68|34.33|         2|        1|     v3|2024-10-01 17:12:00|
|       2|    367|Phoenix|     FR|  609.46|13.61|         2|        2|     v2|2024-02-03 12:06:00|
|       3|    184|Chicago|     DE|  108.61|73.86|         1|        2|  


Saved as Parquet: 10,000 rows


## Summary



In [8]:

print("""
Nested JSON patterns:
  Struct access  : df.select("parent.child.grandchild")
  All fields     : F.col("parent.*")
  Explode array  : F.explode("items") → one row per element
  With position  : F.posexplode("items") → (pos, element)
  Keep nulls     : F.explode_outer("items") → null row for empty/null arrays
  Embedded JSON  : F.from_json(col, schema) → StructType
  Struct to JSON : F.to_json(col) → StringType
  JSONPath extract: F.get_json_object(col, "$.key.subkey")
  Multi-extract  : F.json_tuple(col, "k1","k2","k3")
""")



Nested JSON patterns:
  Struct access  : df.select("parent.child.grandchild")
  All fields     : F.col("parent.*")
  Explode array  : F.explode("items") → one row per element
  With position  : F.posexplode("items") → (pos, element)
  Keep nulls     : F.explode_outer("items") → null row for empty/null arrays
  Embedded JSON  : F.from_json(col, schema) → StructType
  Struct to JSON : F.to_json(col) → StringType
  JSONPath extract: F.get_json_object(col, "$.key.subkey")
  Multi-extract  : F.json_tuple(col, "k1","k2","k3")

